# Normal distribution

In this lab, we'll investigate the probability distribution that is most central to statistics,

> **The normal distribution**.

If we are confident that our data are nearly normal, that opens the door to many powerful statistical methods. Here we'll use the graphical tools of Python to assess the normality of a dataset and also learn how to generate random numbers from a normal distribution.

## The data

Here we'll be working with measurements of body dimensions. This data set contains measurements from 247 men and 260 women, most of whom were considered healthy young adults.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import io
import requests
from plotnine import *

df_url = 'https://raw.githubusercontent.com/akmand/datasets/master/openintro/bdims.csv'
url_content = requests.get(df_url, verify=False).content
bdims = pd.read_csv(io.StringIO(url_content.decode('utf-8')))

Let's take a quick peek at the first few rows of the data.

In [ ]:
pd.set_option('display.max_columns', None)
print(bdims.shape)
bdims.head()

You'll see that for every observation we have 21 measurements, many of which are either diameters or girths. A key to the variable names can be found [here](https://www.openintro.org/book/statdata/?data=bdims), but we'll be focusing on just two columns to get started:

- height in cm (`hgt`), and
- `sex` (1 indicates male, 0 indicates female).

Since men and women tend to have different body dimensions, it will be useful to create two additional data sets: one with only men and another one with only women.

In [ ]:
mdims = bdims[bdims['sex'] == 1]
fdims = bdims[bdims['sex'] == 0]

### Exercise 1
Make a histogram of men's heights and a histogram of women's heights. How would you compare the various aspects of the two distributions?

In [ ]:
(
    ggplot(mdims) +
    aes(x = 'hgt') +
    geom_histogram(bins = 8) +
    labs(title = "Histogram of men's height")
)

In [ ]:
# Adjust the code above to make a histogram of women's heights. Don't forget to change the title!


## The normal distribution

In your description of the distributions, did you use words like *bell-shaped or normal*? It's tempting to say so when faced with a unimodal symmetric distribution.

To see how accurate that description is, we can plot a normal distribution curve on top of a histogram to see how closely the data follow a normal distribution. This normal curve should have the same mean and standard deviation as the data. We'll be working with women's heights, so let's store them as a separate object and then calculate some statistics that will be referenced later.

In [ ]:
fhgtmean = fdims['hgt'].mean()
fhgtsd = fdims['hgt'].std()

Next we make a proportion histogram to use as the backdrop for a normal probability curve.

The difference between a frequency histogram and a proportion histogram is that, while in a frequency histogram the heights of the bars add up to the total number of observations, in a proportion histogram the areas of the bars add up to 1. The area of each bar can be calculated as simply the height times the width of the bar. Using a proportion histogram allows us to properly overlay a normal distribution curve over the histogram since the curve is a normal probability density function.

Frequency and proportion histograms both display the same exact shape; they only differ in their y-axis. You can verify this by comparing the frequency histogram you constructed earlier and the proportion histogram created by the commands below.

In [ ]:
(
    ggplot(fdims) +
    aes(x = 'hgt', y = after_stat('density')) +
    geom_histogram(bins = 8) +
    geom_density(color = 'red') # This overlays the probability distribution.
)

### Exercise 2
Based on the this plot, does it appear that the data follow a nearly normal distribution?

## Evaluating the normal distribution

Eyeballing the shape of the histogram is one way to determine if the data appear to be nearly normally distributed, but it can be frustrating to decide just how close the histogram is to the curve. An alternative approach involves constructing a **normal probability plot**, also called a **normal Q-Q plot** for "quantile-quantile".

> [Here is a 10-minute video explaining the basiscs of Q-Q plots](https://youtu.be/smJBsZ4YQZw?si=251Ursvu7nmHW1Ym)


The code below defines a function that will take a dataframe, variable, and plot title to produce a Q-Q plot.

In [ ]:
def qq_plot(dataframe, variable, title):
  plot = (
      ggplot(dataframe) +
      aes(sample = variable) +
      stat_qq() +
      stat_qq_line() +
      labs(title = title)
  )
  return plot

We can use this `qq_plot` function to produce a Q-Q plot for women's height as follows.

In [ ]:
qq_plot(fdims, 'hgt', "Q-Q plot for women's height")

A data set that is nearly normal will result in a probability plot where the points closely follow the line. Any deviations from normality leads to deviations of these points from the line. The plot for female heights shows points that tend to follow the line but with some errant points towards the tails. We're left with the same problem that we encountered with the histogram above: how close is close enough?

A useful way to address this question is to rephrase it as: what do probability plots look like for data that I know came from a normal distribution? We can answer this by simulating data from a normal distribution using `numpy.random.normal()`.

In [ ]:
sim_norm = np.random.normal(size = len(fdims['hgt']), loc = fhgtmean, scale = fhgtsd)
sim_norm_df = pd.DataFrame(sim_norm, columns = ['hgt'])

The first argument indicates how many numbers you'd like to generate, which we specify to be the same number of heights in the  `fdims` data set using the `len()` function. The last two arguments (`loc` and `scale`) determine the mean and standard deviation of the normal distribution from which the simulated sample will be generated. The last line turns the data into a dataframe (as opposed to a list or array).

We can take a look at the shape of our simulated data set,  `sim_norm_df`, as well as its normal probability plot.

### Exercise 3
Make a normal probability plot of the dataframe `sim_norm_df`.

* Do all of the points fall on the line?
* How does this plot compare to the probability plot for the real data?

In [ ]:
# Fill in the code with the appropriate dataframe!

qq_plot(???, 'hgt', "Simulated Q-Q plot")

Even better than comparing the original plot to a single plot generated from a normal distribution is to compare it to many more plots using the following code. Note, this code will output 9 separate plots---scroll through to compare them!

In [ ]:
fsim_norms_list = [np.random.normal(size = len(fdims['hgt']), loc = fhgtmean, scale = fhgtsd) for i in range (8)]

plot = qq_plot(fdims, 'hgt', "Q-Q plot for women's height")

plot_0 = qq_plot(pd.DataFrame(fsim_norms_list[0], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 1")
plot_1 = qq_plot(pd.DataFrame(fsim_norms_list[1], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 2")
plot_2 = qq_plot(pd.DataFrame(fsim_norms_list[2], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 3")
plot_3 = qq_plot(pd.DataFrame(fsim_norms_list[3], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 4")
plot_4 = qq_plot(pd.DataFrame(fsim_norms_list[4], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 5")
plot_5 = qq_plot(pd.DataFrame(fsim_norms_list[5], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 6")
plot_6 = qq_plot(pd.DataFrame(fsim_norms_list[6], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 7")
plot_7 = qq_plot(pd.DataFrame(fsim_norms_list[7], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 8")

plot_list = [plot, plot_0, plot_1, plot_2, plot_3, plot_4, plot_5, plot_6, plot_7]
for plot in plot_list:
  plot.show()

### Exercise 4
Does the normal probability plot for <code>fdims['hgt']</code> look similar to the plots created for the simulated data? That is, do the plots provide evidence that the female heights are nearly normal?

### Exercise 5
Using the same technique, determine whether or not male heights appear to come from a normal distribution.

In [ ]:
# First, calculate the male heights mean and standard deviation as mhgtmean and mhgtsd.
# Check the code above Exercise 2 for a hint if you need it!

mhgtmean = ???
mhgtsd = ???

In [ ]:
# Next, run this cell to compare the Q-Q plot for men's height to 8 simulated plots, similar to
# Exercise 4.

msim_norms_list = [np.random.normal(size = len(mdims['hgt']), loc = mhgtmean, scale = mhgtsd) for i in range (8)]

plot = qq_plot(mdims, 'hgt', "Q-Q plot for men's height")

plot_0 = qq_plot(pd.DataFrame(msim_norms_list[0], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 1")
plot_1 = qq_plot(pd.DataFrame(msim_norms_list[1], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 2")
plot_2 = qq_plot(pd.DataFrame(msim_norms_list[2], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 3")
plot_3 = qq_plot(pd.DataFrame(msim_norms_list[3], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 4")
plot_4 = qq_plot(pd.DataFrame(msim_norms_list[4], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 5")
plot_5 = qq_plot(pd.DataFrame(msim_norms_list[5], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 6")
plot_6 = qq_plot(pd.DataFrame(msim_norms_list[6], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 7")
plot_7 = qq_plot(pd.DataFrame(msim_norms_list[7], columns = ['hgt']), 'hgt', "Simulated Q-Q plot 8")

plot_list = [plot, plot_0, plot_1, plot_2, plot_3, plot_4, plot_5, plot_6, plot_7]
for plot in plot_list:
  plot.show()

## Normal probabilities

Okay, so now you have a slew of tools to judge whether or not a variable is normally distributed. Why should we care?

It turns out that statisticians know a lot about the normal distribution. Once we decide that a random variable is approximately normal, we can answer all sorts of questions about that variable related to probability. Take, for example, the question of

> "What is the probability that a randomly chosen young adult female is taller than 6 feet (about 182 cm)?"

(The study that published this data set is clear to point out that the sample was not random and therefore inference to a general population is not suggested. We do so here only as an exercise.)

If we assume that female heights are normally distributed (a very close approximation is also okay), we can find this probability by calculating a Z score and consulting a Z table (also called a *normal probability table*). In Python, this is done in one step with the function `norm.cdf()` from `scipy.stats`.

In [ ]:
from scipy.stats import norm

prob_using_Z = 1 - norm.cdf(182, loc = fhgtmean, scale = fhgtsd)
round(prob_using_Z, 4)

Note that the function `norm.cdf()` gives the area under the normal curve below a given value, with a given mean and standard deviation. Since we're interested in the probability that someone is taller than 182 cm, we have to take one minus that probability.

Assuming a normal distribution has allowed us to calculate a theoretical probability. If we want to calculate the probability empirically, we simply need to determine how many observations fall above 182 then divide this number by the total sample size.

In [ ]:
prob_using_empirical = (sum(fdims['hgt'] > 182) / len(fdims['hgt']))
round(prob_using_empirical, 4)

Although the probabilities are not exactly the same, they are reasonably close. In general,

> **The closer that your distribution is to being normal, the more accurate the theoretical probabilities will be.**

### Exercise 6
Write out a probability question that you would like to answer regarding female heights.

Calculate the the probabilities using both the theoretical normal distribution as well as the empirical distribution and note the difference between the results.

### Exercise 7
Write out a probability question that you would like to answer regarding male heights. (You can use the same question you came up with in Exercise 6, if you want).

Calculate the the probabilities using both the theoretical normal distribution as well as the empirical distribution and note the difference between the results.

### Exercise 8

- Did the male heights or the female heights have a closer agreement between the theoretical and empirical methods of calculating probability? Or, was the difference between the theoretical and empirical probability calculations about the same for both male heights and female heights?
- Do you think your answer to the previous question might depend on the quetions(s) you asked/answered in Exercises 6 and 7? Why or why not?
- Does the accuracy of the theoretical probability depend on how closely the data follows the normal distribution? Why or why not?

---

This lab was adapted by Timothy L. Clark, derivative of [OpenIntro Statistics by Diez, Çetinkaya-Rundel, and Barr](https://www.openintro.org/book/os/), released under [Creative Commons BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/deed.en) license.